# 🌐 PATH B1: Self-Host Gemma-4 E4B trên **Google Colab** + ngrok Tunnel

- **LoRA Adapter:** `hung2903/gemma-4-E4B-vaccine-xai-merged`
- **Base Model:** `unsloth/gemma-4-E4B-it`
- **FastAPI + ngrok server**

> ⚠️ Yêu cầu: Runtime → Change runtime type → **T4 GPU**


In [1]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q fastapi uvicorn pyngrok nest-asyncio


In [ ]:
# 🔑 Cấu hình Secrets và Tokens (Google Colab)
# Vui lòng vào biểu tượng "🔑" (Secrets) ở thanh công cụ bên trái Colab,
# thêm hai khóa: "HF_TOKEN" và "NGROK_TOKEN" với giá trị thực tế của bạn,
# sau đó bật quyền truy cập Notebook cho cả hai khóa này.

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    print("✅ Đã nạp thành công HF_TOKEN và NGROK_TOKEN từ Colab Secrets!")
except Exception as e:
    print("⚠️ Không thể nạp tự động từ Colab Secrets. Bạn có thể khai báo thủ công dưới đây:")
    HF_TOKEN = "YOUR_HF_TOKEN"
    NGROK_TOKEN = "YOUR_NGROK_TOKEN"


In [2]:
import torch
import os

# ❌ ĐÃ XOÁ: os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
# ModelScope yêu cầu package + auth riêng, không cần thiết trên Colab.
# HuggingFace Hub hoạt động bình thường trên Colab.

if not torch.cuda.is_available():
    print("❌ LỖI: Không tìm thấy GPU!")
    print("Vào Runtime → Change runtime type → Chọn T4 GPU → Save.")
else:
    # ✅ FIX: Gemma-4 E4B dùng FastModel, KHÔNG phải FastLanguageModel.
    # FastLanguageModel là class cũ cho LLaMA/Mistral, không hỗ trợ Gemma-4.
    from unsloth import FastModel
    from unsloth.chat_templates import get_chat_template

    LORA_ADAPTER = "hung2903/gemma-4-E4B-vaccine-xai-merged"
    MAX_SEQ_LENGTH = 2048

    print("⏳ Loading Gemma-4 E4B Merged Model...")
    model, tokenizer = FastModel.from_pretrained(
        model_name=LORA_ADAPTER,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        token=HF_TOKEN,
    )
    FastModel.for_inference(model)
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

    print(f"✅ Loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    _device = next(model.parameters()).device
    print(f"Model device: {_device}")

    # ✅ FIX: Warmup ĐẶT TRONG else — tránh NameError nếu không có GPU.
    # Chạy trong main thread trước khi khởi động FastAPI để tránh
    # CUDA threadpool deadlock khi uvicorn spawn worker threads.
    print("Warming up model...")
    dummy_enc = tokenizer(text="test", return_tensors="pt").to(_device)
    with torch.no_grad():
        _ = model.generate(
            input_ids=dummy_enc["input_ids"],
            attention_mask=dummy_enc["attention_mask"],
            max_new_tokens=15,
            use_cache=True,
        )
    print("✅ Model warmed up successfully!")


✅ Secrets loaded from Colab Userdata


In [5]:
# ═══════════════════════════════════════════════════════════════
# Parser ANSWER-FIRST v3 — đồng bộ logic với các notebook khác
# ═══════════════════════════════════════════════════════════════
import re as _re

REV_MISINFO_ORDER = [
    ('khong tin gia',1),('không tin giả',1),('khong sai',1),('không sai',1),
    ('dung su that',1),('đúng sự thật',1),('chinh xac',1),('chính xác',1),
    ('accurate',1),('khong lien quan',1),('không liên quan',1),
    ('tin gia',0),('tin giả',0),('tin sai lech',0),('tin sai lệch',0),
    ('misinformation',0),('sai su that',0),('sai sự thật',0),
]
REV_STANCE = {
    'ung ho':0,'ủng hộ':0,'support':0,'tan thanh':0,'tán thành':0,
    'phan doi':1,'phản đối':1,'oppose':1,'chong':1,'chống':1,
    'trung lap':2,'trung lập':2,'neutral':2,
}
REV_SENTIMENT = {
    'tieu cuc':0,'tiêu cực':0,'negative':0,
    'trung tinh':1,'trung tính':1,'trung hin':1,'neutral':1,
    'trung lap':1,'trung lập':1,
    'tich cuc':2,'tích cực':2,'positive':2,
}

LABEL_MISINFO  = {0: 'Tin giả', 1: 'Chính xác'}
LABEL_STANCE   = {0: 'Ủng hộ', 1: 'Phản đối', 2: 'Trung lập'}
LABEL_SENTIMENT= {0: 'Tiêu cực', 1: 'Trung tính', 2: 'Tích cực'}

def build_prompt(text: str) -> list:
    """Trả về messages list theo format Gemma-4 multimodal (content là list of dicts)."""
    content = (
        f'Phân tích nội dung sau về vaccine:\n\n"{text[:1000]}"\n\n'
        'QUY TẮC PHÂN LOẠI (bắt buộc tuân thủ):\n'
        '- Misinformation CHỈ được chọn 1 trong 2: "Tin gia" hoặc "Chinh xac". '
        'TUYỆT ĐỐI KHÔNG dùng từ khác. Nội dung không liên quan vaccine '
        'hoặc không có thông tin sai = "Chinh xac".\n'
        '- Stance CHỈ: "Ung ho" / "Phan doi" / "Trung lap".\n'
        '- Sentiment CHỈ: "Tieu cuc" / "Trung tinh" / "Tich cuc".\n\n'
        'Trả lời theo ĐÚNG cấu trúc: KẾT QUẢ trước, GIẢI THÍCH sau.\n'
        '=== KẾT QUẢ ===\n'
        '- Misinformation: <Tin gia HOẶC Chinh xac>\n'
        '- Stance: <Ung ho HOẶC Phan doi HOẶC Trung lap>\n'
        '- Sentiment: <Tieu cuc HOẶC Trung tinh HOẶC Tich cuc>\n'
        '=== GIẢI THÍCH ===\n'
        '<lý luận chi tiết bằng tiếng Việt>'
    )
    # ✅ Gemma-4 multimodal: content là list of typed dicts (KHÔNG phải plain string)
    return [{"role": "user", "content": [{"type": "text", "text": content}]}]

def _mis_binary(seg):
    s = (seg or '').strip().lower()
    if not s: return -1, False
    for conj in [' nhưng ', ' tuy nhiên ', ' nhưng,', ' song ']:
        if conj in s:
            s = s.split(conj, 1)[1]
            break
    NEG_WORDS = ['không', 'khong', 'chẳng', 'chả ', 'chưa ', 'no ']
    for kw, val in REV_MISINFO_ORDER:
        idx = s.find(kw)
        if idx == -1: continue
        prefix = s[:idx]
        has_neg = any(neg in prefix for neg in NEG_WORDS)
        if has_neg:
            return (1 - val), True
        return val, True
    return 1, True

def _match_longest(seg, rev):
    s = (seg or '').lower()
    for k in sorted(rev, key=len, reverse=True):
        if k in s:
            return rev[k], True
    return -1, False

def _seg_after_keyword(line, keyword):
    """
    Lấy đoạn QUYẾT ĐỊNH sau từ khoá task — BỎ ngoặc đơn TIÊU ĐỀ ngay sau keyword.
    VD: '**Misinformation (Tin gia):** Nội dung chính xác' → 'Nội dung chính xác'
    """
    low = line.lower()
    idx = low.find(keyword.rstrip(':'))
    if idx == -1: return None
    after_kw = line[idx + len(keyword.rstrip(':')):]
    after_kw_clean = _re.sub(r'^[*\s]*\([^)]*\)', '', after_kw)
    if ':' in after_kw_clean:
        decision = after_kw_clean.split(':', 1)[1]
    else:
        decision = after_kw_clean
    for marker in ['->', '→', 'kết luận:', 'ket luan:', 'phân loại:', 'phan loai:']:
        if marker in decision.lower():
            decision = decision.lower().split(marker, 1)[1][:120]
            break
    mt = _re.search(r'\[([^\]]{1,40})\]', decision)
    if mt: return mt.group(1)
    return decision[:200]

def _parse_block(block_text):
    lines = [l.strip() for l in block_text.split('\n')]
    def find(pfx):
        for ln in lines:
            if pfx in ln: return ln.split(pfx, 1)[-1]
        return None
    seg_m  = find('misinformation:')
    seg_st = find('stance:')
    seg_se = find('sentiment:')
    n = sum(1 for x in [seg_m, seg_st, seg_se] if x is not None)
    m,  mo  = _mis_binary(seg_m)          if seg_m  is not None else (-1, False)
    st, so  = _match_longest(seg_st, REV_STANCE)    if seg_st is not None else (-1, False)
    se, seo = _match_longest(seg_se, REV_SENTIMENT) if seg_se is not None else (-1, False)
    return m, st, se, (mo and so and seo), n

def parse_output(text):
    t = (text or "")
    t_low = t.lower()
    # LAYER 1: block KẾT QUẢ chuẩn
    markers = list(_re.finditer(
        r'(?:=== kết quả ===|=== ket qua ===|^kết quả:|^ket qua:)',
        t_low, _re.M))
    if markers:
        blocks = []
        for i, mt in enumerate(markers):
            start = mt.end()
            end = markers[i+1].start() if i+1 < len(markers) else len(t_low)
            seg = t_low[start:end]
            for gm in ['=== giải thích ===','=== giai thich ===','giải thích:','giai thich:']:
                if gm in seg:
                    seg = seg.split(gm, 1)[0]; break
            blocks.append(seg)
        best = None
        for blk in blocks:
            res = _parse_block(blk)
            if best is None or (res[3] and not best[3]) or \
               (res[3]==best[3] and res[4] > best[4]):
                best = res
        if best and best[3]:
            return best[0], best[1], best[2], True
    # LAYER 2: văn xuôi — dùng _seg_after_keyword (bỏ ngoặc đơn tiêu đề task)
    lines = t.split('\n')
    seg_m = seg_st = seg_se = None
    for ln in lines:
        if 'misinformation' in ln.lower() and seg_m  is None and len(ln) < 300:
            seg_m  = _seg_after_keyword(ln, 'misinformation:')
        if 'stance'         in ln.lower() and seg_st is None and len(ln) < 300:
            seg_st = _seg_after_keyword(ln, 'stance:')
        if 'sentiment'      in ln.lower() and seg_se is None and len(ln) < 300:
            seg_se = _seg_after_keyword(ln, 'sentiment:')
    m,  mo  = _mis_binary(seg_m)          if seg_m  is not None else (-1, False)
    st, so  = _match_longest(seg_st, REV_STANCE)    if seg_st is not None else (-1, False)
    se, seo = _match_longest(seg_se, REV_SENTIMENT) if seg_se is not None else (-1, False)
    if mo and so and seo:
        return m, st, se, True
    # LAYER 3: accept ≥2/3
    if (mo + so + seo) >= 2:
        return m, st, se, True
    return m, st, se, False

# Sanity tests
assert _mis_binary(' tin gia')[0] == 0
assert _mis_binary(' chinh xac')[0] == 1
assert _mis_binary(' khong tin gia')[0] == 1
print("✅ Parser v3 ANSWER-FIRST loaded")
print("   • Layer 1: block === KẾT QUẢ ===")
print("   • Layer 2: văn xuôi + bỏ ngoặc đơn tiêu đề task")
print("   • Layer 3: accept ≥2/3 nhãn")


✅ Parser v3 ANSWER-FIRST loaded
   • Layer 1: block === KẾT QUẢ ===
   • Layer 2: văn xuôi + bỏ ngoặc đơn tiêu đề task
   • Layer 3: accept ≥2/3 nhãn


In [6]:
def generate_reasoning(text: str, max_tokens: int = 350) -> dict:
    """
    Sinh reasoning và parse nhãn cấu trúc.
    Trả về dict: {reasoning, misinfo, stance, sentiment, parsed}
    """
    # build_prompt đã trả về messages với content dạng Gemma-4 multimodal
    messages = build_prompt(text)

    # Tách 2 bước tokenize: apply_chat_template(tokenize=False) + tokenizer(text=...)
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    _device = next(model.parameters()).device
    model_inputs = tokenizer(text=formatted_text, return_tensors="pt").to(_device)
    inputs       = model_inputs["input_ids"]
    attn_mask    = model_inputs["attention_mask"]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            attention_mask=attn_mask,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            use_cache=True,
            repetition_penalty=1.2,
        )

    raw = tokenizer.decode(
        outputs[0][inputs.shape[-1]:], skip_special_tokens=True
    ).strip()
    raw = raw.replace("<end_of_turn>", "").replace("<|turn>", "").strip()

    # Parse nhãn cấu trúc
    m, st, se, parsed = parse_output(raw)

    # Tách phần GIẢI THÍCH làm reasoning text
    if "=== GIẢI THÍCH ===" in raw:
        reasoning = raw.split("=== GIẢI THÍCH ===")[-1].strip()
    elif "=== giải thích ===" in raw.lower():
        idx = raw.lower().find("=== giải thích ===")
        reasoning = raw[idx:].split('\n', 1)[-1].strip()
    else:
        reasoning = raw
    if not reasoning.startswith("Lý luận"):
        reasoning = "Lý luận: " + reasoning

    return {
        "reasoning":  reasoning,
        "misinfo":    LABEL_MISINFO.get(m, "?"),
        "stance":     LABEL_STANCE.get(st, "?"),
        "sentiment":  LABEL_SENTIMENT.get(se, "?"),
        "parsed":     parsed,
        "raw":        raw,
    }

# Quick test
test_result = generate_reasoning("Vắc-xin COVID gây vô sinh ở phụ nữ trẻ.")
print("=== TEST ===")
print(f"Misinfo:   {test_result['misinfo']}")
print(f"Stance:    {test_result['stance']}")
print(f"Sentiment: {test_result['sentiment']}")
print(f"Parsed:    {test_result['parsed']}")
print(f"Reasoning: {test_result['reasoning'][:200]}...")


=== TEST ===
Misinfo:   Tin giả
Stance:    Phản đối
Sentiment: Tiêu cực
Parsed:    True
Reasoning: Lý luận: **Phân tích:**

1.  **Misinformation (Tin giả):** Khẳng định vắc-xin COVID gây vô sinh ở phụ nữ trẻ là một tuyên bố y khoa hoàn toàn không có cơ sở khoa học và đã bị các tổ chức y tế lớn trên...


In [7]:
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok, conf
import nest_asyncio
import uvicorn
import threading
import subprocess
import time
import torch

# Kill port if occupied
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(1)

conf.get_default().auth_token = NGROK_TOKEN
app = FastAPI(title="VaccineNLP Gemma-4 Inference Server")

# Global Thread Lock to prevent multi-threaded FastAPI threadpool CUDA deadlock on Colab
model_lock = threading.Lock()

class InferenceRequest(BaseModel):
    text: str
    max_tokens: int = 350
    # Note: T4 inference ~80-120s

class InferenceResponse(BaseModel):
    reasoning:  str
    misinfo:    str = "?"
    stance:     str = "?"
    sentiment:  str = "?"
    parsed:     bool = False
    status:     str = "success"
    error:      str = ""

@app.post("/predict", response_model=InferenceResponse)
def predict(req: InferenceRequest):
    try:
        if not req.text or not req.text.strip():
            return InferenceResponse(reasoning="", status="error", error="Empty text")
        
        # Global lock to synchronize GPU generation requests safely
        with model_lock:
            result = generate_reasoning(req.text, req.max_tokens)
            # Clear GPU cache after decoding to prevent memory fragmentation
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            return InferenceResponse(
                reasoning = result["reasoning"],
                misinfo   = result["misinfo"],
                stance    = result["stance"],
                sentiment = result["sentiment"],
                parsed    = result["parsed"],
            )
    except Exception as e:
        return InferenceResponse(reasoning="", status="error", error=str(e))

@app.get("/health")
def health():
    return {
        "status":     "healthy",
        "model":      "Gemma-4 E4B (VaccineNLP fine-tuned)",
        "gpu_mem_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
    }

@app.get("/")
def root():
    return {"app": "VaccineNLP Gemma-4 Self-host", "version": "2.0"}

nest_asyncio.apply()

print("Starting ngrok tunnel...")
public_url = ngrok.connect(8000, "http").public_url
print(f"\n{'='*60}")
print(f" PUBLIC URL:      {public_url}")
print(f" PREDICT:         {public_url}/predict")
print(f" HEALTH:          {public_url}/health")
print(f"{'='*60}\n")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)
print("Server running. Keep this cell alive!")

🌐 Starting ngrok tunnel...

🔗 PUBLIC URL:      https://pearle-staglike-nonsyntonically.ngrok-free.dev
🔗 PREDICT:         https://pearle-staglike-nonsyntonically.ngrok-free.dev/predict
🔗 HEALTH:          https://pearle-staglike-nonsyntonically.ngrok-free.dev/health

▲ SAO CHÉP URL TRÊN — dùng cho HF Spaces config
✅ Server running. Keep this cell alive!


In [9]:
import requests

try:
    response = requests.post(
        f"{public_url}/predict",
        json={"text": "Vắc-xin COVID gây vô sinh ở phụ nữ trẻ và biến đổi gen ở trẻ em."},
        timeout=300,  # Tăng lên 300s để tránh timeout trên T4
    )

    print(f"Status: {response.status_code}")
    if response.status_code == 200:
        d = response.json()
        print(f"\n📊 Misinformation : {d['misinfo']}")
        print(f"📊 Stance         : {d['stance']}")
        print(f"📊 Sentiment      : {d['sentiment']}")
        print(f"📊 Parsed OK      : {d['parsed']}")
        print(f"\n💭 Reasoning:\n{d['reasoning']}")
    else:
        print(f"❌ Error: {response.text}")

except requests.exceptions.Timeout:
    print("❌ Lỗi: Server không phản hồi trong thời gian quy định (Timeout).")
    print("Kiểm tra xem cell Server (ngrok) có còn đang chạy hay không.")
except Exception as e:
    print(f"❌ Đã xảy ra lỗi: {e}")

Status: 200

📊 Misinformation : Tin giả
📊 Stance         : Phản đối
📊 Sentiment      : Tiêu cực
📊 Parsed OK      : True

💭 Reasoning:
Lý luận: **Phân tích:**

1. **Misinformation (Tin giả):** Câu nói "Vắc-xin COVID gây vô sinh ở phụ nữ trẻ và biến đổi gen ở trẻ em" là một tuyên bố hoàn toàn không có cơ sở khoa học. Các tổ chức y tế uy tín trên thế giới như WHO, CDC và Bộ Y tế Việt Nam đều khẳng định rằng vắc-xin COVID-19 an toàn cho cả phụ nữ trong độ tuổi sinh sản và trẻ em, không gây vô sinh hay thay đổi gen. Đây là dạng thông tin sai lệch nhằm gieo rắc nỗi sợ hãi. $\rightarrow$ **Tin gia**.

2. **Stance (Lập trường):** Người đăng tải câu này đang đưa ra những cáo buộc nghiêm trọng về tác hại của vaccine để cảnh báo người khác không nên tiêm. $\rightarrow$ **Phan doi**.

3. **Sentiment (Sắc thái):** Việc đưa ra những nhận định cực đoan và đáng sợ như "gây vô sinh" và "biến đổi gen" mang tính chất công kích, tiêu cực đối với chương trình tiêm chủng. $\rightarrow$ **Tieu cuc**.

=== KẾ

In [10]:
import time, requests

print("🟢 Server alive — waiting for requests...")
print(f"🔗 {public_url}/predict")
print("⚠️  URL ngrok thay đổi theo session — cập nhật HF Spaces Secrets nếu restart")
print("🛑 Stop session để tắt")

while True:
    time.sleep(60)
    try:
        h = requests.get(f"{public_url}/health", timeout=10).json()
        print(f"[{time.strftime('%H:%M:%S')}] Alive | GPU: {h['gpu_mem_gb']} GB")
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] ⚠️ {e}")


🟢 Server alive — waiting for requests...
🔗 https://pearle-staglike-nonsyntonically.ngrok-free.dev/predict
⚠️  URL ngrok thay đổi theo session — cập nhật HF Spaces Secrets nếu restart
🛑 Stop session để tắt
[15:24:22] Alive | GPU: 11.28 GB
[15:25:23] Alive | GPU: 11.28 GB
[15:26:25] Alive | GPU: 11.28 GB
[15:27:28] Alive | GPU: 11.28 GB
[15:28:29] Alive | GPU: 11.28 GB
[15:29:31] Alive | GPU: 11.28 GB
[15:30:31] Alive | GPU: 11.28 GB
[15:31:32] Alive | GPU: 11.28 GB
[15:32:33] Alive | GPU: 11.28 GB
[15:33:34] Alive | GPU: 11.28 GB
[15:34:34] Alive | GPU: 11.28 GB
[15:35:36] Alive | GPU: 11.28 GB
[15:36:39] Alive | GPU: 11.28 GB
[15:37:41] Alive | GPU: 11.28 GB
[15:38:41] Alive | GPU: 11.28 GB
[15:39:46] Alive | GPU: 11.28 GB
[15:40:46] Alive | GPU: 11.28 GB
[15:41:46] Alive | GPU: 11.28 GB
[15:42:50] Alive | GPU: 11.28 GB
[15:43:50] Alive | GPU: 11.28 GB
[15:44:51] Alive | GPU: 11.28 GB
[15:45:52] Alive | GPU: 11.35 GB
[15:46:52] Alive | GPU: 11.4 GB
[15:47:55] Alive | GPU: 11.44 GB
[15:

KeyboardInterrupt: 